# Experiment 11: Unsupervised Anomaly Detection
## Isolation Forest & Local Outlier Factor on Public Cellular Network KPI Data

**Project:** AI-Based Network Anomaly & Fault-Risk Detection for Telecom Networks  
**Research Question:** How does unsupervised anomaly detection compare with supervised classification for detecting network anomalies in public telecom KPI data, and which KPI features contribute most to detection?  
**Dataset Reference:** Hugging Face `AliMaatouk/TelecomTS` (32,000 public cellular traces, 64 statistical moments)  
**Research Scope Notice:** Independent research prototype using public cellular KPI data. The dataset does NOT represent SLTMobitel operational records, Sri Lankan telecom infrastructure, or confirmed physical fault causes.

---

### Objectives
1. Implement **Isolation Forest (IF)**: an unsupervised tree ensemble based on path length isolation.
2. Implement **Local Outlier Factor (LOF)**: an unsupervised density-based method comparing local reachability density.
3. Fit both models strictly **without anomaly labels**.
4. Generate anomaly scores, decision functions, and discrete predictions.
5. Evaluate predictions against reference test labels on the exact same 80/20 test split ($N=6,400$).
6. Investigate the effect of the contamination hyperparameter and neighborhood size ($k$).

In [ ]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, roc_curve, precision_recall_curve
)

# Step 1: Load precomputed 64 features
pickle_path = "../data/processed/telecom_features_64.pkl"
if not os.path.exists(pickle_path):
    pickle_path = "data/processed/telecom_features_64.pkl"

with open(pickle_path, "rb") as f:
    X, y = pickle.load(f)

# Step 2: Deterministic 80/20 stratified partition
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Loaded: X_train={X_train.shape}, X_test={X_test.shape}")
print(f"Test Class Balance: Normal={sum(y_test==0)}, Anomaly={sum(y_test==1)} ({y_test.mean()*100:.2f}%)")

## Part 1: Isolation Forest (iForest)
### Theoretical Background
Isolation Forest isolates anomalies by recursively partitioning randomly selected features at random cut-points.
Because anomalies are few and structurally divergent, they require fewer random orthogonal splits to isolate from the majority population, resulting in shorter average path lengths in the isolation trees:
$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$
where $E(h(x))$ is the expected path length and $c(n)$ is the average path length of unsuccessful searches in a Binary Search Tree.

Crucially, **no target labels ($y$) are provided during model fitting**.

In [ ]:
# Fit Isolation Forest (Unsupervised)
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.0386, # matches training anomaly rate for balanced cutoff
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train)

# Invert decision function: lower scikit-learn scores mean abnormal; inversion ensures higher = more anomalous
iso_scores = -iso_forest.decision_function(X_test)
iso_preds = np.where(iso_forest.predict(X_test) == -1, 1, 0)

tn, fp, fn, tp = confusion_matrix(y_test, iso_preds).ravel()
print("Isolation Forest Test Metrics:")
print(f"  Precision: {precision_score(y_test, iso_preds)*100:.2f}%")
print(f"  Recall:    {recall_score(y_test, iso_preds)*100:.2f}%")
print(f"  F1-Score:  {f1_score(y_test, iso_preds)*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_score(y_test, iso_scores)*100:.2f}%")
print(f"  PR-AUC:    {average_precision_score(y_test, iso_scores)*100:.2f}%")
print(f"  Confusion: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

## Part 2: Local Outlier Factor (LOF)
### Theoretical Background
Local Outlier Factor measures the local deviation of a given data point with respect to its surrounding neighbors.
It calculates:
1. $k$-distance and reachability distance to neighbors.
2. Local reachability density (lrd).
3. The ratio of local reachability density of neighbors to that of the point itself (LOF score).

To evaluate on unseen test observations ($X_{test}$), we employ scikit-learn's **novelty detection mode (`novelty=True`)**, fitting on standardized training data.

In [ ]:
# Standardize features for distance-based k-NN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lof_novelty = LocalOutlierFactor(
    n_neighbors=20,
    novelty=True,
    contamination=0.0386,
    n_jobs=-1
)
lof_novelty.fit(X_train_scaled)

lof_scores = -lof_novelty.decision_function(X_test_scaled)
lof_preds = np.where(lof_novelty.predict(X_test_scaled) == -1, 1, 0)

tn, fp, fn, tp = confusion_matrix(y_test, lof_preds).ravel()
print("Local Outlier Factor (k=20) Test Metrics:")
print(f"  Precision: {precision_score(y_test, lof_preds)*100:.2f}%")
print(f"  Recall:    {recall_score(y_test, lof_preds)*100:.2f}%")
print(f"  F1-Score:  {f1_score(y_test, lof_preds)*100:.2f}%")
print(f"  ROC-AUC:   {roc_auc_score(y_test, lof_scores)*100:.2f}%")
print(f"  PR-AUC:    {average_precision_score(y_test, lof_scores)*100:.2f}%")
print(f"  Confusion: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

## Part 3: Contamination & Sensitivity Sweeps
We analyze how the contamination hyperparameter in Isolation Forest and the neighborhood size ($k$) in LOF influence detection rates.

In [ ]:
# Load and display saved sensitivity sweeps
results_dir = "../results" if os.path.exists("../results") else "results"
iso_sens = pd.read_csv(os.path.join(results_dir, "isolation_forest_contamination_sensitivity.csv"))
lof_sens = pd.read_csv(os.path.join(results_dir, "lof_k_sensitivity.csv"))

print("Isolation Forest Contamination Sweep:")
print(iso_sens)

print("\nLocal Outlier Factor Neighborhood Size (k) Sweep:")
print(lof_sens)